# Day 28 — AWS Bedrock for deployment

Bedrock is a managed way to call foundation models (Claude included) from inside AWS: one API,
IAM auth instead of API keys, data that stays in your account and region, and add-ons
(Guardrails, Knowledge Bases, provisioned throughput). Today: what it gives you over the
first-party API, the `converse` request shape, model IDs and inference profiles, the pricing
model, and when to pick which door.

Runnable against a mock `bedrock-runtime` client; the real `boto3` calls are shown alongside.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | Three doors to Claude on AWS | 5 min |
| 1 | What Bedrock adds over the direct API | 10 min |
| 2 | The `converse` API (and `invoke_model`) | 14 min |
| 3 | Model IDs, regions, inference profiles | 8 min |
| 4 | Pricing: on-demand vs provisioned throughput | 10 min |
| 5 | Guardrails, Knowledge Bases, Agents | 8 min |
| 6 | Exercises + quiz | 3 min |

Kernel: **Python (ai-upskill)**.

In [1]:
import json, time
print("ready")

ready


## 0 — Three doors to Claude on AWS (5 min)

| Door | Auth | Data path | Pricing | Use when |
| ---- | ---- | --------- | ------- | -------- |
| **First-party Anthropic API** | API key / OAuth | to `api.anthropic.com` | Anthropic list price | fastest features, simplest, not AWS-bound |
| **Claude Platform on AWS** (Anthropic-operated) | AWS | in AWS, Anthropic-operated | Anthropic list price | want AWS billing + same-day feature parity |
| **Amazon Bedrock** (AWS-operated) | AWS IAM | in your AWS account/region | AWS Bedrock pricing (separate) | already deep in AWS: IAM, VPC, CloudTrail, one API across model vendors |

This lesson is the **Bedrock** door. The tradeoff vs first-party: you get AWS-native
everything, at the cost of a lag on the newest models/features and a different (sometimes
higher) price sheet.

## 1 — What Bedrock adds over the direct API (10 min)

- **No API keys.** Auth is IAM — roles, policies, temporary credentials, `AssumeRole`. A
  Lambda calls Bedrock with its execution role; nothing to rotate.
- **Data stays put.** Requests don't leave your AWS region; you're not sending prompts to a
  third-party endpoint. Inputs/outputs aren't used to train models.
- **VPC / PrivateLink.** Call Bedrock over a private endpoint, no internet egress.
- **CloudTrail + CloudWatch.** Every `InvokeModel` call is logged for audit; metrics and model
  invocation logging (to S3/CloudWatch) out of the box.
- **One API, many vendors.** `converse` works the same for Claude, Llama, Titan, Mistral,
  Nova — swap `modelId`, keep the code.
- **Provisioned throughput.** Reserve capacity for predictable high volume (§4).
- **Guardrails, Knowledge Bases, Agents.** Managed content filtering, managed RAG, managed
  tool-use loops (§5).
- **AWS Marketplace billing.** One bill, existing enterprise agreement, credits.

What you give up: newest models land on Bedrock weeks-to-months after the first-party API;
some features (fast mode, certain betas) are first-party only; pricing differs.

## 2 — The `converse` API (14 min)

Bedrock has two request styles:

- **`converse` / `converse_stream`** — a unified, vendor-neutral message API (recommended).
- **`invoke_model` / `invoke_model_with_response_stream`** — you send the vendor's *native*
  JSON body (for Claude, the Anthropic Messages shape). More control, less portability.

### `converse` (boto3)

```python
import boto3
brt = boto3.client("bedrock-runtime", region_name="us-east-1")

resp = brt.converse(
    modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",   # an inference profile id
    system=[{"text": "You are a terse assistant."}],
    messages=[{"role": "user", "content": [{"text": "Capital of France?"}]}],
    inferenceConfig={"maxTokens": 512, "temperature": 0.0, "stopSequences": []},
)
print(resp["output"]["message"]["content"][0]["text"])
print(resp["usage"])        # {'inputTokens': ..., 'outputTokens': ..., 'totalTokens': ...}
print(resp["stopReason"])   # 'end_turn' | 'max_tokens' | 'tool_use' | 'stop_sequence' | ...
```

Note the shape differs from the first-party SDK: `content` is a list of `{"text": ...}` /
`{"toolUse": ...}` blocks, keys are camelCase, `system` is a list of `{"text": ...}`.

In [2]:
# a mock bedrock-runtime client mirroring converse()
class MockBedrockRuntime:
    def __init__(self, region_name="us-east-1"): self.region = region_name
    def converse(self, *, modelId, messages, system=None, inferenceConfig=None, toolConfig=None):
        cfg = inferenceConfig or {}
        last = messages[-1]["content"][0].get("text", "")
        # tool use branch
        if toolConfig and "weather" in last.lower():
            return {"output": {"message": {"role": "assistant", "content": [
                        {"toolUse": {"toolUseId": "tu1", "name": "get_weather", "input": {"city": "Paris"}}}]}},
                    "stopReason": "tool_use",
                    "usage": {"inputTokens": 120, "outputTokens": 15, "totalTokens": 135}}
        reply = f"[bedrock:{modelId.split('.')[-1][:20]}] answer to: {last[:40]}"
        out_tok = len(reply) // 4
        stop = "max_tokens" if out_tok > cfg.get("maxTokens", 512) else "end_turn"
        return {"output": {"message": {"role": "assistant", "content": [{"text": reply}]}},
                "stopReason": stop,
                "usage": {"inputTokens": 90, "outputTokens": out_tok, "totalTokens": 90 + out_tok},
                "metrics": {"latencyMs": 480}}
    def converse_stream(self, **kw):
        r = self.converse(**{k: v for k, v in kw.items() if k != "toolConfig"})
        text = r["output"]["message"]["content"][0]["text"]
        yield {"messageStart": {"role": "assistant"}}
        for i in range(0, len(text), 8):
            yield {"contentBlockDelta": {"delta": {"text": text[i:i+8]}, "contentBlockIndex": 0}}
        yield {"messageStop": {"stopReason": r["stopReason"]}}
        yield {"metadata": {"usage": r["usage"], "metrics": r.get("metrics", {})}}

brt = MockBedrockRuntime()
resp = brt.converse(
    modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system=[{"text": "Be terse."}],
    messages=[{"role": "user", "content": [{"text": "Capital of Japan?"}]}],
    inferenceConfig={"maxTokens": 256})
print(resp["output"]["message"]["content"][0]["text"])
print("usage:", resp["usage"], "| stop:", resp["stopReason"])

[bedrock:claude-sonnet-4-5-20] answer to: Capital of Japan?
usage: {'inputTokens': 90, 'outputTokens': 14, 'totalTokens': 104} | stop: end_turn


In [3]:
# tool use via converse: toolConfig instead of `tools`, camelCase keys
TOOL_CONFIG = {"tools": [{"toolSpec": {
    "name": "get_weather", "description": "Current weather for a city.",
    "inputSchema": {"json": {"type": "object", "properties": {"city": {"type": "string"}},
                             "required": ["city"]}}}}]}

def bedrock_tool_loop(question, max_steps=3):
    messages = [{"role": "user", "content": [{"text": question}]}]
    for _ in range(max_steps):
        r = brt.converse(modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
                         messages=messages, toolConfig=TOOL_CONFIG)
        msg = r["output"]["message"]; messages.append(msg)
        if r["stopReason"] != "tool_use":
            return msg["content"][0]["text"]
        results = []
        for block in msg["content"]:
            if "toolUse" in block:
                tu = block["toolUse"]
                out = {"city": tu["input"]["city"], "tempC": 14}       # run the tool
                results.append({"toolResult": {"toolUseId": tu["toolUseId"],
                                               "content": [{"json": out}]}})
        messages.append({"role": "user", "content": results})
    return "stopped"

print(bedrock_tool_loop("what's the weather like?"))

[bedrock:claude-sonnet-4-5-20] answer to: 


In [4]:
# streaming via converse_stream
acc = ""
for ev in brt.converse_stream(modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        messages=[{"role": "user", "content": [{"text": "explain bedrock in one line"}]}],
        inferenceConfig={"maxTokens": 200}):
    if "contentBlockDelta" in ev:
        acc += ev["contentBlockDelta"]["delta"]["text"]
    elif "metadata" in ev:
        usage = ev["metadata"]["usage"]
print(acc, "\n(usage:", usage, ")")

[bedrock:claude-sonnet-4-5-20] answer to: explain bedrock in one line 
(usage: {'inputTokens': 90, 'outputTokens': 17, 'totalTokens': 107} )


### Or: the Anthropic SDK's Bedrock client

If you want the *first-party SDK surface* (same code as Day 22–24) but routed through Bedrock:

```python
from anthropic import AnthropicBedrockMantle
client = AnthropicBedrockMantle(aws_region="us-east-1")
client.messages.create(model="anthropic.claude-sonnet-4-5", max_tokens=512,
                       messages=[{"role": "user", "content": "hi"}])
```

Bedrock model IDs there take an `anthropic.` prefix. This is the least-friction path if you
already wrote against the Anthropic SDK.

## 3 — Model IDs, regions, inference profiles (8 min)

- **Foundation model ID:** `anthropic.claude-sonnet-4-5-20250929-v1:0` — pinned to one region.
- **Inference profile ID:** `us.anthropic.claude-sonnet-4-5-20250929-v1:0` — the `us.` (or
  `eu.`, `apac.`) prefix means **cross-region inference**: AWS routes your request to whichever
  region in that geo has capacity. Higher availability, same price. **Use inference profiles**
  for production unless you have a hard data-residency reason to pin one region.
- Not every model is in every region. Check `bedrock:ListFoundationModels`.
- You must **request model access** in the console once per account before first use.
- `bedrock` client = control plane (list models, manage provisioned throughput, guardrails);
  `bedrock-runtime` client = data plane (`converse`, `invoke_model`).

In [5]:
# control-plane vs data-plane, and discovering models
BEDROCK_CLIENTS = {
 "bedrock":         ["list_foundation_models", "get_foundation_model_availability",
                     "create_provisioned_model_throughput", "create_guardrail",
                     "create_model_invocation_logging_configuration"],
 "bedrock-runtime": ["converse", "converse_stream", "invoke_model",
                     "invoke_model_with_response_stream", "apply_guardrail"],
 "bedrock-agent-runtime": ["retrieve", "retrieve_and_generate", "invoke_agent"],
}
for c, ops in BEDROCK_CLIENTS.items():
    print(f"{c:22s} -> {', '.join(ops)}")

bedrock                -> list_foundation_models, get_foundation_model_availability, create_provisioned_model_throughput, create_guardrail, create_model_invocation_logging_configuration
bedrock-runtime        -> converse, converse_stream, invoke_model, invoke_model_with_response_stream, apply_guardrail
bedrock-agent-runtime  -> retrieve, retrieve_and_generate, invoke_agent


## 4 — Pricing: on-demand vs provisioned throughput (10 min)

- **On-demand:** pay per input/output token, no commitment. Great default. Subject to
  account-level throughput quotas (requests/min, tokens/min) — you can raise them via a quota
  request.
- **Provisioned Throughput:** reserve *model units* (a fixed tokens/min capacity) for 1-month
  or 6-month commitments. Predictable latency and no throttling at high volume, but you pay
  for the reservation whether you use it or not. Only worth it above a steady, high load.
- **Batch inference:** ~50% off for large asynchronous jobs (results to S3).
- **Prompt caching** is available on Bedrock for supported Claude models — same idea as Day 22
  (cache a stable prefix), separate cache-write / cache-read token rates.

Bedrock's per-token prices for Claude differ from Anthropic's first-party list — check the
Bedrock pricing page. Below: a decision calculator.

In [6]:
# Illustrative numbers (check the live pricing page). $ per 1M tokens.
ON_DEMAND = dict(pin=3.0, pout=15.0)                 # e.g. a Sonnet-tier model
BATCH     = dict(pin=1.5, pout=7.5)                  # ~50% off, async
PROVISIONED_MODEL_UNIT_MONTHLY = 40_000              # $ / model unit / month (1-mo commit)
MODEL_UNIT_TOKENS_PER_MIN = 400_000                  # capacity of one unit

def on_demand_cost(calls_per_month, in_tok, out_tok):
    return calls_per_month * (in_tok*ON_DEMAND["pin"] + out_tok*ON_DEMAND["pout"]) / 1e6

def provisioned_cost(peak_tokens_per_min):
    units = -(-peak_tokens_per_min // MODEL_UNIT_TOKENS_PER_MIN)   # ceil
    return units * PROVISIONED_MODEL_UNIT_MONTHLY, units

for calls, label in [(200_000, "small"), (5_000_000, "medium"), (40_000_000, "large")]:
    od = on_demand_cost(calls, 800, 250)
    peak_tpm = calls/30/24/60 * (800+250) * 3          # rough: 3x average as peak
    pv, units = provisioned_cost(peak_tpm)
    print(f"{label:7s} {calls:>11,} calls/mo:  on-demand ${od:>12,.0f}   "
          f"provisioned ${pv:>10,.0f} ({units} units)  -> use {'provisioned' if pv < od else 'on-demand'}")

small       200,000 calls/mo:  on-demand $       1,230   provisioned $    40,000 (1.0 units)  -> use on-demand
medium    5,000,000 calls/mo:  on-demand $      30,750   provisioned $    40,000 (1.0 units)  -> use on-demand
large    40,000,000 calls/mo:  on-demand $     246,000   provisioned $   320,000 (8.0 units)  -> use on-demand


Rule: **start on-demand.** Move to provisioned throughput only when (a) your load is steady
and high, (b) you're hitting throttling, or (c) provisioned math beats on-demand at your
sustained volume — and even then, size for p95 load, not peak. Use batch for anything that
doesn't need a synchronous answer.

## 5 — Guardrails, Knowledge Bases, Agents (8 min)

Bedrock bundles managed versions of things you built earlier this course:

| Feature | What it is | You built this on... |
| ------- | ---------- | -------------------- |
| **Guardrails** | configurable filters: denied topics, PII redaction, profanity, prompt-injection detection, "grounded-ness" checks; applied to input and/or output; callable standalone via `apply_guardrail` | Day 16 (refusal), Day 25 (safety), Day 27 (redaction) |
| **Knowledge Bases** | managed RAG: point it at S3, it chunks + embeds (Titan/Cohere) + stores (OpenSearch Serverless / Aurora pgvector / Pinecone) + serves via `retrieve` / `retrieve_and_generate` | Weeks 5–6 |
| **Agents** | managed tool-use loop: you define action groups (Lambda functions) + a knowledge base, Bedrock runs the plan→act→observe loop | Day 19, Day 21 |
| **Flows** | visual pipeline builder wiring prompts, KBs, Lambdas, conditions | Day 20 (orchestration) |

The tradeoff is the same as always: managed = less code, faster start, less control and
portability. A Knowledge Base is the fastest way to a working RAG endpoint; a hand-built
pipeline (Day 18) is the way to tune chunking, hybrid retrieval, and reranking exactly.

In [7]:
# Knowledge Base retrieve_and_generate -- the managed-RAG one-liner
RAG_CALL = '''
bar = boto3.client("bedrock-agent-runtime", region_name="us-east-1")
resp = bar.retrieve_and_generate(
    input={"text": "How long do refunds take?"},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": "KB1234ABCD",
            "modelArn": "arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-sonnet-4-5-20250929-v1:0",
        },
    },
)
print(resp["output"]["text"])
for c in resp["citations"]:
    for ref in c["retrievedReferences"]:
        print(" source:", ref["location"]["s3Location"]["uri"])
'''
print(RAG_CALL)
print("-> chunk/embed/store/retrieve/generate/cite, all managed. Trade tuning for time-to-ship.")


bar = boto3.client("bedrock-agent-runtime", region_name="us-east-1")
resp = bar.retrieve_and_generate(
    input={"text": "How long do refunds take?"},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": "KB1234ABCD",
            "modelArn": "arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-sonnet-4-5-20250929-v1:0",
        },
    },
)
print(resp["output"]["text"])
for c in resp["citations"]:
    for ref in c["retrievedReferences"]:
        print(" source:", ref["location"]["s3Location"]["uri"])

-> chunk/embed/store/retrieve/generate/cite, all managed. Trade tuning for time-to-ship.


## 6 — Exercises

1. **converse ↔ Messages mapping.** Write `to_converse(anthropic_messages)` and
   `from_converse(bedrock_resp)` that translate between the first-party shape (Day 22) and the
   `converse` shape (list-of-`{"text":...}`, camelCase, `system` as a list). Round-trip a
   2-turn conversation.
2. **Provisioned break-even.** Solve for the calls/month at which provisioned beats on-demand,
   as a function of the peak-to-average ratio (1.5, 3, 5). Plot it.
3. **Batch vs sync.** For a nightly job of 2M documents to classify (300 in / 20 out each),
   compare on-demand sync cost vs 50%-off batch. What's the latency you trade away?
4. **Guardrail as a standalone check.** Model `apply_guardrail(content, source)` as a function
   that flags denied topics and redacts emails. Run it on 5 inputs; show it works
   independently of any model call (useful for pre-screening).
5. **Region/profile choice.** Given a data-residency requirement of "EU only", which model ID
   form do you use and why can't you use the `us.` inference profile? What availability do you
   give up?
6. **Door decision.** For each: (a) a startup already on Anthropic's API wanting to add
   Bedrock later; (b) a bank requiring VPC-only egress and CloudTrail; (c) a team wanting the
   newest Claude the day it ships. Which door, and the main tradeoff.

> **Attempt every exercise and the quiz first.** The worked solutions and the answer key live in [`solutions/solutions.ipynb`](solutions/solutions.ipynb) — open it only to check your work, not to start.

## Self-check quiz


1. Name three things Bedrock gives you that the first-party API doesn't.
2. What's the main thing you give up by using Bedrock?
3. `converse` vs `invoke_model` — when would you use each?
4. What is a cross-region inference profile and why use one in production?
5. On-demand vs provisioned throughput — the decision rule.
6. You need managed RAG on Bedrock with the least code. Which feature, and what do you trade?
7. `bedrock` client vs `bedrock-runtime` client — what's each for?

## Where this goes next

- **Day 29 — Serving & cost:** whichever door you pick, you have to size and pay for the
  serving layer — throughput, batching, concurrency, autoscaling, caching, and a runnable cost
  model.